# 02. 모델링 전 가설 검증 (H1, H2) — Batch 1 전용

- 목적: 01_EDA에서 나온 가설 중 모델 전략을 정하는 데 가장 중요한 두 가지를 **Batch 1 교차검증만으로** 확인.
- Batch 2·3는 읽지 않음 (Batch 2는 최종 평가 1회 전용).
- 사전 준비: `.venv/bin/python src/preprocess.py 1` (캐시 `data/cache/batch1_*`)

| # | 가설 | 근거(01_EDA) |
|---|---|---|
| **H1** | `dQ_log_var` 1개만으로도 보조 feature(`Tavg_mean`, `chargetime_mean`, `c_avg`)를 넣은 모델과 CV MAPE가 거의 같다 | 보조 feature는 배치마다 부호·크기가 불안정, `c_avg`는 `dQ_log_var`와 r=0.92로 겹침 |
| **H2** | 정책 단위 그룹 CV의 점수가 셀 단위 랜덤 CV보다 나쁘다(랜덤 CV가 낙관적). 정책 정보를 담은 feature·유연한 모델일수록 차이가 크다 | 정책당 셀이 1~3개라 같은 정책이 train/valid에 섞이면 누수 |

**검증 방법**: 5-fold, 20회 반복(분할 시드 42부터). 각 반복에서 5개 fold의 out-of-fold 예측을 모아 MAPE 1개를 계산 → 20개 MAPE의 평균·표준편차로 비교. 랜덤 CV는 셀 단위로 섞고, 그룹 CV는 정책(23종)을 fold에 무작위로 배정(같은 정책은 항상 같은 fold).

In [1]:
import numpy as np, pandas as pd, re
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

SEED, N_SPLIT, N_REP = 42, 5, 20

d = pd.read_csv('../data/cache/batch1_summary.csv'); z = np.load('../data/cache/batch1_dq.npz')
cells = d.drop_duplicates('cell')[['cell', 'cycle_life', 'policy']].reset_index(drop=True)
dq = z['q100'] - z['q10']
cells['dQ_log_var'] = np.log10(np.abs(np.nanvar(dq, axis=1)))
e = d[d.cycle <= 100].groupby('cell')
cells = cells.join(pd.DataFrame({'Tavg_mean': e.Tavg.mean(), 'chargetime_mean': e.chargetime.mean()}), on='cell')
def c_avg(p):
    a, soc, b = map(float, re.match(r'([\d.]+)C\((\d+)%\)-([\d.]+)C', p).groups()); return a*soc/100 + b*(1-soc/100)
cells['c_avg'] = cells.policy.map(c_avg)
y = cells.cycle_life.values; groups = cells.policy.values
print(f'셀 {len(cells)}개, 정책 {cells.policy.nunique()}종 (정책당 셀 수: {cells.groupby("policy").size().value_counts().sort_index().to_dict()})')
cells.head()

셀 46개, 정책 23종 (정책당 셀 수: {1: 1, 2: 21, 3: 1})


,cell,cycle_life,policy,dQ_log_var,Tavg_mean,chargetime_mean,c_avg
0,0,1190.0,3.6C(80%)-3.6C,-5.014861,31.287710,13.250388,3.6
1,1,1179.0,3.6C(80%)-3.6C,-5.013960,31.017011,13.240949,3.6
2,2,1177.0,3.6C(80%)-3.6C,-4.737000,31.164788,13.245097,3.6
3,3,1226.0,4C(80%)-4C,-4.442613,29.642777,11.932744,4.0
4,4,1227.0,4C(80%)-4C,-4.647744,31.134395,11.932781,4.0


In [2]:
def mape(y, p): return np.mean(np.abs(y - p) / y)

def splits(kind, rep):
    """한 반복의 (train_idx, valid_idx) 목록. random=셀 단위, group=정책 단위."""
    if kind == 'random':
        return list(KFold(N_SPLIT, shuffle=True, random_state=SEED + rep).split(y))
    rng = np.random.RandomState(SEED + rep); pols = rng.permutation(np.unique(groups))
    fold_of = {p: i % N_SPLIT for i, p in enumerate(pols)}; f = np.array([fold_of[g] for g in groups])
    return [(np.where(f != k)[0], np.where(f == k)[0]) for k in range(N_SPLIT)]

def cv_mape(make_model, cols, kind):
    """N_REP회 반복한 out-of-fold MAPE (반복별 1개씩)."""
    X = cells[cols].values if cols else np.zeros((len(cells), 1)); out = []
    for r in range(N_REP):
        pred = np.zeros(len(y))
        for tr, va in splits(kind, r):
            m = make_model(); m.fit(X[tr], y[tr]); pred[va] = m.predict(X[va])
        out.append(mape(y, pred))
    return np.array(out)

class MedianBaseline:                       # 기준선: 학습 fold의 수명 중앙값
    def fit(self, X, y): self.m = np.median(y); return self
    def predict(self, X): return np.full(len(X), self.m)

linear = lambda: make_pipeline(StandardScaler(), LinearRegression())
forest = lambda: RandomForestRegressor(n_estimators=300, random_state=SEED)
DQ = ['dQ_log_var']; AUX = ['Tavg_mean', 'chargetime_mean', 'c_avg']

## H1. `dQ_log_var` 1개 vs 보조 feature 추가 (그룹 CV, 정책 단위)

판정 기준(미리 정함): 보조 feature를 넣은 모델의 평균 MAPE가 `dQ_log_var` 단독보다 **반복 간 표준편차보다 크게** 낮고, 20회 중 대부분(≥ 15회)에서 더 좋을 때만 "추가 효과 있음".

In [3]:
sets = {'dQ 단독': DQ, 'dQ + Tavg': DQ + ['Tavg_mean'], 'dQ + chargetime': DQ + ['chargetime_mean'], 'dQ + c_avg': DQ + ['c_avg'], 'dQ + 보조 3개': DQ + AUX, 'c_avg 단독': ['c_avg']}
base = cv_mape(linear, DQ, 'group'); rows = []
for name, cols in sets.items():
    s = cv_mape(linear, cols, 'group'); diff = s - base
    rows.append({'feature': name, '평균 MAPE': s.mean(), '반복 표준편차': s.std(), 'dQ 단독 대비 차이': diff.mean(), '더 좋은 반복 수(/20)': int((diff < 0).sum())})
rows.append({'feature': '(참고) 중앙값 기준선', '평균 MAPE': cv_mape(lambda: MedianBaseline(), None, 'group').mean(), '반복 표준편차': np.nan, 'dQ 단독 대비 차이': np.nan, '더 좋은 반복 수(/20)': np.nan})
h1 = pd.DataFrame(rows).set_index('feature'); h1.style.format({'평균 MAPE': '{:.1%}', '반복 표준편차': '{:.2%}', 'dQ 단독 대비 차이': '{:+.2%}'}, na_rep='-')

,평균 MAPE,반복 표준편차,dQ 단독 대비 차이,더 좋은 반복 수(/20)
feature,,,,
dQ 단독,9.2%,0.39%,+0.00%,0.000000
dQ + Tavg,9.5%,0.36%,+0.33%,0.000000
dQ + chargetime,9.8%,0.43%,+0.61%,0.000000
dQ + c_avg,8.8%,0.37%,-0.33%,16.000000
dQ + 보조 3개,9.4%,0.41%,+0.22%,6.000000
c_avg 단독,9.7%,0.41%,+0.51%,4.000000
(참고) 중앙값 기준선,19.5%,-,-,-


**결과 · 해석 (H1)** — 그룹 CV, 평균 MAPE
- `dQ_log_var` 단독 **9.2%** (중앙값 기준선 19.5%의 절반 이하). `Tavg`(9.5%), `chargetime`(9.8%), 보조 3개 전부(9.4%)를 추가하면 오히려 **나빠짐** → 이 feature들은 추가 효과 없음(가설 지지).
- 예외는 `c_avg` 추가(8.8%): 0.33%p 낮고 20회 중 16회에서 더 좋음. 사전 기준(반복 표준편차 0.37%p보다 크게 낮아야 함)에는 **약간 못 미치는 경계 수준**.
- 해석: `c_avg`가 Batch 1 안에서 약간 도움이 되는 것은 EDA 결과(Batch 1 r=−0.87)와 일치하지만, 이 이득이 Batch 1에만 있는 관계일 가능성이 큼(Batch 2 −0.25, 같은 정책이 배치마다 수명이 다름). Batch 1 CV만으로는 일반화를 판단할 수 없으므로 **보조 feature는 Batch 1 CV에서 이득이 확실할 때만 추가**한다는 원칙을 유지하고, `c_avg`는 "경계 수준, 일반화 근거 없음"으로 제외.
- `c_avg` 단독도 9.7%로 `dQ_log_var` 단독(9.2%)보다 나쁨 → 충전 정책보다 셀에서 측정한 ΔQ 신호가 더 낫다는 EDA 결론과 일치.

## H2. 랜덤 CV vs 정책 단위 그룹 CV

모델 3개를 두 분할로 평가. 같은 정책 셀이 train과 valid에 섞이면 정책 정보(`c_avg` 등)가 누수되므로, 그런 feature를 쓰는 모델과 유연한 모델(랜덤 포레스트)에서 랜덤 CV 점수가 더 좋게(낙관적으로) 나올 것으로 예상.

In [4]:
models = {'선형 (dQ 단독)': (linear, DQ), '선형 (dQ + 보조 3개)': (linear, DQ + AUX), '랜덤 포레스트 (dQ + 보조 3개)': (forest, DQ + AUX), '랜덤 포레스트 (dQ 단독)': (forest, DQ)}
rows = []
for name, (mk, cols) in models.items():
    r_, g_ = cv_mape(mk, cols, 'random'), cv_mape(mk, cols, 'group')
    rows.append({'모델': name, '랜덤 CV MAPE': r_.mean(), '그룹 CV MAPE': g_.mean(), '그룹 − 랜덤': (g_ - r_).mean(), '그룹이 더 나쁜 반복 수(/20)': int((g_ > r_).sum())})
h2 = pd.DataFrame(rows).set_index('모델'); h2.style.format({'랜덤 CV MAPE': '{:.1%}', '그룹 CV MAPE': '{:.1%}', '그룹 − 랜덤': '{:+.2%}'})

,랜덤 CV MAPE,그룹 CV MAPE,그룹 − 랜덤,그룹이 더 나쁜 반복 수(/20)
모델,,,,
선형 (dQ 단독),8.5%,9.2%,+0.65%,20
선형 (dQ + 보조 3개),8.6%,9.4%,+0.77%,19
랜덤 포레스트 (dQ + 보조 3개),7.7%,9.1%,+1.33%,20
랜덤 포레스트 (dQ 단독),7.8%,8.9%,+1.08%,17


**결과 · 해석 (H2)**
- 네 모델 모두 **그룹 CV가 랜덤 CV보다 나쁨**(MAPE +0.65~+1.33%p, 20회 중 17~20회). 랜덤 CV는 낙관적 → **정책 단위 그룹 CV를 쓰기로 한 결정이 맞음.**
- 랜덤 포레스트(+1.1~+1.3%p)가 선형(+0.65~+0.77%p)보다 차이가 큼 → 유연한 모델일수록 낙관 편향이 큼(가설 지지).
- "정책 feature를 넣을수록 차이가 커진다"는 약함: 보조 feature 유무에 따른 차이의 차이가 0.1~0.25%p로 반복 표준편차(약 0.4%p) 수준.
- 한계: 그룹을 정책 문자열 단위로 나눴지만 `5.4C(40%)-3.6C`와 `5.4C(50%)-3.6C`처럼 비슷한 정책이 서로 다른 fold에 들어가면 일부 누수가 남음 → 그룹 CV도 여전히 약간 낙관적일 수 있음.
- 주의: Batch 1 안의 CV로는 **학습 범위 밖 외삽 문제가 드러나지 않음**. 랜덤 포레스트(dQ 단독, 8.9%)가 선형(9.2%)보다 오히려 좋게 나온 것은 Batch 1 범위 안에서의 결과이며, 트리가 불리하다는 가설(H4)은 이 검증으로 판단할 수 없음.

## 요약

| 가설 | 판정 | 모델 전략에 주는 의미 |
|---|---|---|
| **H1** `dQ_log_var` 1개로 충분 | 대체로 지지 (`c_avg`만 경계 수준의 작은 이득, 일반화 근거 없음) | 입력은 `dQ_log_var` 1개로 시작 유지 |
| **H2** 랜덤 CV가 낙관적 | 지지 (4개 모델 모두 그룹 CV가 더 나쁨, 랜덤 포레스트에서 더 큼) | 정책 단위 그룹 CV를 사용 |

- 이 노트북은 **Batch 1만** 사용. Batch 2는 한 번도 읽지 않음.
- 남은 가설(H3 log 타깃, H4 외삽, H5 장수명 과소예측)은 DAY2 모델링에서 확인.